# Feature Engineering — Energy Demand (t+1 Prediction)

## Objective

Prepare features for predicting **next-day energy demand (t+1)**.

We construct two datasets:

1. **Load-only features**
   - Lagged demand
   - Calendar features

2. **Load + Weather features**
   - Lagged demand
   - Calendar
   - Temperature

---

## Goal

Evaluate whether:

- Model complexity improves prediction
- Or additional information (weather) reduces the need for complex models

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd
import numpy as np

from configs.config import *

In [2]:
file_path = PROCESSED_DATA_DIR / "aep_with_weather.csv"

df = pd.read_csv(file_path)

df["Datetime"] = pd.to_datetime(df["Datetime"])

df.head()

,Datetime,AEP_MW,temp_mean
0,2004-10-01,14284.521739,16.0
1,2004-10-02,12999.875000,15.6
2,2004-10-03,12227.083333,10.9
3,2004-10-04,14309.041667,13.0
4,2004-10-05,14439.708333,9.5


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5055 entries, 0 to 5054
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Datetime   5055 non-null   datetime64[us]
 1   AEP_MW     5055 non-null   float64       
 2   temp_mean  5055 non-null   float64       
dtypes: datetime64[us](1), float64(2)
memory usage: 118.6 KB


In [4]:
for lag in [1, 2, 3, 7]:
    df[f"load_lag_{lag}"] = df["AEP_MW"].shift(lag)

df.head()

,Datetime,AEP_MW,temp_mean,load_lag_1,load_lag_2,load_lag_3,load_lag_7
0,2004-10-01,14284.521739,16.0,NaN,NaN,NaN,NaN
1,2004-10-02,12999.875000,15.6,14284.521739,NaN,NaN,NaN
2,2004-10-03,12227.083333,10.9,12999.875000,14284.521739,NaN,NaN
3,2004-10-04,14309.041667,13.0,12227.083333,12999.875000,14284.521739,NaN
4,2004-10-05,14439.708333,9.5,14309.041667,12227.083333,12999.875000,NaN


In [5]:
df["target"] = df["AEP_MW"].shift(-1)

df.head()

,Datetime,AEP_MW,temp_mean,load_lag_1,load_lag_2,load_lag_3,load_lag_7,target
0,2004-10-01,14284.521739,16.0,NaN,NaN,NaN,NaN,12999.875000
1,2004-10-02,12999.875000,15.6,14284.521739,NaN,NaN,NaN,12227.083333
2,2004-10-03,12227.083333,10.9,12999.875000,14284.521739,NaN,NaN,14309.041667
3,2004-10-04,14309.041667,13.0,12227.083333,12999.875000,14284.521739,NaN,14439.708333
4,2004-10-05,14439.708333,9.5,14309.041667,12227.083333,12999.875000,NaN,14424.791667


In [6]:
df_model = df.dropna().reset_index(drop=True)

df_model.head()

,Datetime,AEP_MW,temp_mean,load_lag_1,load_lag_2,load_lag_3,load_lag_7,target
0,2004-10-08,14350.333333,16.0,14449.416667,14424.791667,14439.708333,14284.521739,12934.541667
1,2004-10-09,12934.541667,17.2,14350.333333,14449.416667,14424.791667,12999.875000,12260.375000
2,2004-10-10,12260.375000,12.5,12934.541667,14350.333333,14449.416667,12227.083333,14299.750000
3,2004-10-11,14299.750000,11.3,12260.375000,12934.541667,14350.333333,14309.041667,14614.916667
4,2004-10-12,14614.916667,11.4,14299.750000,12260.375000,12934.541667,14439.708333,14454.416667


In [17]:
df_model["day_of_week"] = df_model["Datetime"].dt.dayofweek
df_model["is_weekend"] = df_model["day_of_week"].isin([5, 6]).astype(int)

In [18]:
cols_to_keep = [
    "Datetime",
    "AEP_MW",
    "temp_mean",
    "load_lag_1",
    "load_lag_2",
    "load_lag_3",
    "load_lag_7",
    "day_of_week",
    "is_weekend",
    "target"
]

df_model = df_model[cols_to_keep].copy()

output_path = PROCESSED_DATA_DIR / "aep_features.csv"
df_model.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: /Users/suvo/Projects/model-complexity/data/processed/aep_features.csv
